# Leakage-safe drought forecasting baselines

This lightweight notebook explains the Phase 2 performance floor. It reads only compact committed evidence; authoritative construction, training, prediction, and validation remain in reusable source modules.

In [ ]:
import csv
import json
from pathlib import Path

import pandas as pd


def repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'reports/phase2e_comparison.json').is_file():
            return candidate
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
ROOT = repository_root()

## 1. Forecasting problem and supplied information

The task predicts next-month Total Water Storage (TWS), a combined measure of surface water, soil water, groundwater, snow, and ice. Inputs include SPEI drought indices, soil moisture, coordinates, and calendar features. An **effective horizon** is the calendar-month distance from the last genuinely observed TWS to the target. Consecutive masks make it range from 1 to 7 even though each row targets its next month.

In [ ]:
comparison = json.loads((ROOT / 'reports/phase2e_comparison.json').read_text(encoding='utf-8'))
phase2c = json.loads((ROOT / 'reports/phase2c_horizon_examples_manifest.json').read_text(encoding='utf-8'))
lightgbm_path = ROOT / next(run['metrics_path'] for run in comparison['runs'] if run['model'] == 'lightgbm_basic')
lightgbm_metrics = json.loads(lightgbm_path.read_text(encoding='utf-8'))
with (ROOT / 'experiments/registry.csv').open(encoding='utf-8', newline='') as stream:
    registry = list(csv.DictReader(stream))
assert len(registry) == len(comparison['runs']) == 7
assert comparison['preferred']['model'] == 'lightgbm_basic'

## 2. Frozen validation, prediction contract, and leakage prevention

A random split could let later months or nearby copies of the same location history inform earlier validation targets—**target leakage**. Validation-v1 instead fits independently at F01 (2004-09) and F02 (2008-12) and pools 551,965 distinct **out-of-fold predictions**, meaning predictions for targets excluded from fitting. The canonical prediction contract fixes keys, calendar relationships, identities, and coverage. Exact-calendar joins ensure a source is truly month `y-h`, never a previous row or later backfill.

## 3. Deterministic baselines and fallbacks

A **persistence baseline** carries the last genuinely observed TWS forward. **Climatology** uses a location's fold-local average for the target calendar month. Seasonal naïve uses the exact same month one year earlier; other references use global mean, location mean, or trend plus seasonal residual. A **fallback** is an explicit lower-priority source used when the preferred source is unavailable. Every chain terminates in a prediction, so difficult rows cannot disappear.

In [ ]:
ranked_comparison = pd.DataFrame([{
 'rank': r['rank'], 'model': r['model'], 'run_id': r['run_id'],
 'pooled_rmse': r['pooled_rmse'], 'fold_gap': r['absolute_fold_gap'],
 'runtime_seconds': r['runtime_seconds'], 'peak_memory_mb': r['peak_memory_mb'],
 'coverage': r['coverage']['fraction'], 'fallback_rate': r['fallback_rate']}
 for r in comparison['runs']]).sort_values('rank').reset_index(drop=True)
ranked_comparison

## 4. Fold and horizon comparisons

**Pooled RMSE** sums squared errors across all out-of-fold rows before taking one square root; it does not average fold RMSEs. Lower is better. Fold gaps describe temporal stability, while horizon slices show the effect of increasingly old observed TWS.

In [ ]:
fold_comparison = pd.DataFrame([{'model': r['model'], 'F01_rmse': r['folds']['F01']['rmse'], 'F01_rows': r['folds']['F01']['count'], 'F02_rmse': r['folds']['F02']['rmse'], 'F02_rows': r['folds']['F02']['count'], 'absolute_gap': r['absolute_fold_gap']} for r in comparison['runs']]).sort_values('absolute_gap')
fold_comparison

In [ ]:
focus = {r['model']: r for r in comparison['runs'] if r['model'] in {'lightgbm_basic', 'persistence'}}
horizon_comparison = pd.DataFrame({model: {int(h): value for h, value in run['horizon_rmse'].items()} for model, run in focus.items()}).rename_axis('effective_horizon')
horizon_comparison

In [ ]:
mask_comparison = pd.DataFrame({model: {state: values['rmse'] for state, values in run['mask_state'].items()} for model, run in focus.items()}).rename_axis('input_TWS_state')
mask_comparison

## 5. Horizon-aware deterministic sampling and gradient-boosted trees

Phase 2C found 9,470,789 eligible examples. Full downstream learning was unsafe, so **deterministic sampling** used one fixed seed and hash ranking to retain 2,000,000 rows in the validation fold/horizon distribution. Each uses covariates from `y-1` and observed TWS from exact `y-h`. LightGBM fits **gradient-boosted decision trees**, where each small tree corrects earlier errors. The fixed benchmark used 200 trees, two threads, 13 approved features, and no early stopping or sweep.

In [ ]:
resource_comparison = ranked_comparison[['model', 'runtime_seconds', 'peak_memory_mb']].copy()
weak_latitude_bands = pd.DataFrame([{'band': k, **v} for k, v in lightgbm_metrics['metrics']['latitude_band'].items() if v['count']]).sort_values('rmse', ascending=False)
spei_rows = [{'feature': feature, 'bin': name, **values} for feature, bins in lightgbm_metrics['metrics']['spei'].items() for name, values in bins.items() if values['count']]
weak_spei_bins = pd.DataFrame(spei_rows).sort_values('rmse', ascending=False).head(8)
resource_comparison, weak_latitude_bands.head(3), weak_spei_bins

## 6. Decision, limitations, and Phase 3 hypotheses

The saved comparison prefers LightGBM because it has the best pooled score, improves both folds, has the smallest fold gap, and covers every row. Persistence remains the strongest deterministic reference and fallback. Measured limitations are worsening long-horizon error, weaker masked-input performance, uneven geographic/SPEI-bin error, capped training, and roughly 2.2 GiB peak evaluation memory. Phase 3 hypotheses—not findings—include horizon-specific calibration and a small number of auditable features for masked and long-horizon cases.

## 7. Authoritative reproduction commands

Accepted artifacts already exist. Do **not** rerun expensive commands merely to view results. From the repository root:

```powershell
# Baseline: about 81-145 s, 2.2-2.3 GiB
.\.venv\Scripts\python.exe -m drought_forecasting.deterministic_baselines --baseline persistence
# Horizon examples: about 232 s, 2.1 GiB
.\.venv\Scripts\python.exe -m drought_forecasting.horizon_examples --config configs\phase2c_horizon_examples.yaml
# Fixed LightGBM: about 125 s, 2.1 GiB
.\.venv\Scripts\python.exe -m drought_forecasting.lightgbm_benchmark --config configs\phase2d_lightgbm.yaml
# Lightweight comparison consolidation
.\.venv\Scripts\python.exe -m drought_forecasting.phase2_comparison
```

Large training, model, and OOF artifacts are Git-ignored. The official `references/official/StarterNotebook.ipynb` is protected and unchanged.